In [ ]:
%load_ext autoreload
%autoreload 2

# Random

## Temp Sandbox

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from colligo.sdk import NativeClient
from colligo.base.worker_registery import set_worker_registery
# from colligo.models.types.firefly_v2.clio import CLIO_V2, ClioExecutor, ClioRequest
from colligo.models.firefly.t5.types import T5_V2, T5Request, T5Executor

from colligo.models.firefly.imcoding.decoder_v3.types import (
    LdmDecoderV3Request,
    # LdmDecoder_V3_32_8x,
    # LDM_DECODER_V3_32_8X
    LdmDecoder_V3_128_8x,
    LDM_DECODER_V3_128_8X
)
from clio.diffusion_core.mixture_of_models_loop import MixtureOfModels
from clio.typings.interface import CoreParameters
from clio.typings.interface import MixtureOfModelsTensors, ConditionTensors

In [ ]:
registery = await NativeClient.create([T5_V2, LDM_DECODER_V3_128_8X])

In [ ]:
set_worker_registery(registery)
request = T5Request(prompts=["A beautiful rose in a nice green garden with a nice sunset, realistic, high-res, photo-realistic, zoomed-out"])
response = await T5Executor.call(request)
txt_tensors = response.embeddings.as_torch()
txt_tensors.shape

In [ ]:
set_worker_registery(registery)
request = T5Request(prompts=["red, ugly, low-res, pixelated, blurry, noisy, bad quality, bad lighting, bad angle, bad composition, bad focus, bad exposure, bad framing, bad contrast, bad saturation"])
response = await T5Executor.call(request)
avoid_txt_tensors = response.embeddings.as_torch()
avoid_txt_tensors.shape

In [ ]:
from inference.inference_utils.model_loading_utils import initialize_eps_model
from inference.inference_utils import yaml_utils

config_yaml = "/mnt/localssd/hshariati/colligo/contrib/AdobeDiffusion/apps/inference/inference/configs/model_explore/dit/dit_seq_inference.yaml"

config = yaml_utils.yaml_to_inference_config(config_yaml)
diffusion_model, device = initialize_eps_model(config)

In [ ]:
params = CoreParameters(
    dimensions=config.dimensions,
    guidance_parameters=config.guidance_parameters,
    sampling_parameters=config.sampling_parameters,
    iteration_parameters=config.iteration_parameters,
)

moe = MixtureOfModels(params=params)

In [ ]:
lst_positive_conditions = [ConditionTensors(text_embedding=txt_tensors)]
lst_negative_conditions = [ConditionTensors(text_embedding=avoid_txt_tensors)]

moe_input = MixtureOfModelsTensors(
    lst_positive_conditions=lst_positive_conditions,
    lst_negative_conditions=lst_negative_conditions,
    t5_text_embedding=None
)

In [ ]:
import torch
samples = moe.dispatch(
    lst_eps_callable=diffusion_model,
    lst_controlnet_adaptors=None,
    lst_eps_device=[device] * 1,
    lst_latent_upsampler_callable=None,
    inputs=moe_input,
    lst_seeds=[0],
    initial_image=torch.zeros(1, 12, 12, 32, 32).to(device),
    params=params,
    binary_mask=None,
    classifier_guidance=None,
    callback=None,
)

In [ ]:
from clio.process_utils.resize_util import resize_t
sample = resize_t(samples.iterates[0], size=(128,128))

In [ ]:
sample.shape

In [ ]:
from colligo.base.types.tensor import Tensor

output_images = []
set_worker_registery(registery)
for im in range(12):
    request = LdmDecoderV3Request(encodings=Tensor.from_torch(sample[im:im+1, :, : ,:]), cast_to_uint8=True)
    decoded_image = await LdmDecoder_V3_128_8x.call(request)
    output_images.append(decoded_image.images.as_torch())

In [ ]:
import torch
from torchvision.transforms import functional

pils = []
for image in output_images:
    pil = functional.to_pil_image(image[0])
    pil = pil.resize((512, 512))
    pil.show()

## Running DiT video Clineto model

In [ ]:
#check point and training config
checkpoint: /sensei-fs/users/kduarte/runai_training_logs/dit_t2v_sequential_model_M_newgenie32_train-2024-04-25T08-36-51/lightning_logs/version_0/checkpoints/epoch=0-step=130000.ckpt
#Training config
config: /sensei-fs/users/kduarte/runai_training_logs/dit_t2v_sequential_model_M_newgenie32_train-2024-04-25T08-36-51/lightning_logs/version_0/config.yaml

In [ ]:
#loading clineto model
from inference.inference_app import Inference
from inference.inference_utils import yaml_utils

config_yaml = "/mnt/localssd/hshariati/colligo/contrib/AdobeDiffusion/apps/inference/inference/configs/model_explore/dit/dit_seq_inference.yaml"

config = yaml_utils.yaml_to_inference_config(config_yaml)

app = Inference(config)

In [ ]:
#inference on video
app.config.dimensions.height = [24]
app.config.dimensions.width = [40]
app.config.dimensions.num_frames = [12]
images = app(text_prompt="Astronaut on a camel")

In [ ]:
#hs writing video to disk and also displaying in notebook
# pip install imageio[ffmpeg] #to install FFMPEG
import imageio
from IPython.display import HTML

temp_video_file = f"./temp_output_video.mp4"
imageio.mimwrite(
    temp_video_file,
    images.images[0],
    format="mp4",
    fps=12,
    codec="libx264",
    output_params=["-pix_fmt", "yuv420p"],
)

video_html = """
<video width="640" height="360" controls>
    <source src="{}" type="video/mp4">
</video>
""".format(temp_video_file)
display(HTML(video_html))

In [ ]:
#inference on images
app.config.dimensions.num_frames = [-1]
images = app(text_prompt="Astronaut on a camel")
images.images[0]

In [ ]:
# model comparison command
clio-model-comparison --config_files ./apps/inference/inference/configs/model_explore/dit/dit_seq_inference.yaml --limit_prompts 10 --model_tags "model_dit_name" --text_prompt_file ./cliocore/prototyping_tools/src/clio/prototyping_tools/data/kineto_eval_sentences/sora_prompts.txt --save_to ./outputs/dit_model --upload_to_s3

## EMA (Exponential Moving Average) on a set of checkpoints

In [ ]:
#This is the script to do EMA (Exponential Moving Average) on a set of checkpoints
/mnt/localssd/hshariati/colligo/vendor/AdobeDiffusion/cliocore/nas/src/clio/nas/offline_ema.py

# example call:
    # python -m clio.offline_ema --expm /sensei-fs/.../your_logging_folder --save_to /sensei-fs/checkpoint_ema.ckpt --num_checkpoints 10 --decay 0.7

# For md we 5 and 0.7 only on G models. Although we can do last 10 as well.

In [ ]:
# debugging the regular experssion for sorting the checkpoints.
# this works for ckpts named: global_step_270000_G.ckpt
import argparse
import copy
from pathlib import Path

import torch

examp = "/sensei-fs/users/hshariati/del/version_19_ckpts"

# load all checkpoints, ignore last.ckpt checkpoints since they are duplicates
all_checkpoints = [
    p.resolve()
    for p in Path(examp).rglob("*.ckpt")
    if p.name != "last.ckpt" and "ema" not in p.name
]

# sort checkpoints first by version then by step within each version folder
all_checkpoints_sorted = sorted(
    all_checkpoints,
    key=lambda x: (
        int(str(x).split("/")[-1].split("_")[-2]),
    ),
)

# return only required checkpoint paths
final = all_checkpoints_sorted[-10 :]

In [ ]:
# If you just run this code on dmd, you get no key "state_dict" error. To resolve this, I added this functon.
def convert_dmd_ckpts(ckpt_paths):
    converted_paths = []
    teacher_path = "/sensei-fs/projects/firefly/clio/training/logs/haravi/c3prod3-27-128px-2680k-aesnorm-mar3/c3prod3-27-128px-2680k-aesnorm-mar3-2024-01-12T21-41-30/lightning_logs/version_0/checkpoints/epoch=0-step=65000.ckpt"
    teacher_unet = torch.load(teacher_path, map_location="cpu")

    for input_path in ckpt_paths:
        input_str = str(input_path)
        unet = torch.load(str(input_str), map_location="cpu")
        keys = list(unet.keys())
        for key in keys:
            if key.startswith("G.eps_model.D"):
                unet[key.replace("G.eps_model.D", "eps_model")] = unet.pop(key)
        unet.pop("g_optimizer")

        for key in teacher_unet["state_dict"]:
            if key in unet:
                teacher_unet["state_dict"][key] = unet[key]

        output_str = input_str.replace(".ckpt", "combined_slimmed.ckpt")
        torch.save(teacher_unet, output_str)
        converted_paths.append(Path(output_str))

    return converted_paths

# also added a way to call it here:
latest_state_dicts = find_latest_k_checkpoints(opt)
converted_dmd_ckpts = convert_dmd_ckpts(latest_state_dicts)
average_state_dict = average_state_dicts(converted_dmd_ckpts, opt.decay)

## Debugging run_inference_clio3_mc.py new (apr 18, 2024)

In [ ]:
# this is the script
/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/scripts/clio3/run_inference_pipelines_clio3.py

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import the functions from script
import sys
sys.path.append("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/scripts/clio3/")
from run_inference_pipelines_clio3 import InferencePipeline

In [ ]:
#this is the class for faking the args
class MockArgs:
    def __init__(self):
        self.config = None
        self.prior_model_id = None
        self.model_id = None
        self.checkpoints_yaml = ""
        self.save_to = None
        self.text_prompt = None
        self.style_prompt = None
        self.style_image = None
        self.structure_image = None
        self.controls = None
        self.prompt_policy = "photo_v3"
        self.use_prior = None
        self.limit_prompts = None
        self.return_file_path = None
        self.aesthetic_score = 6.0
        self.prior_aes = None
        self.face_size = None
        self.use_ar = False
        self.use_adherence = False
        self.device = None
        self.start_idx = None
        self.end_idx = None
        self.run_id = None
        self.use_rag = False

In [ ]:
# add args needed for dmd inference and load

mock_args = MockArgs()
# you can set the device if you want first gpu is 0:
mock_args.device = 1

# mock_args.checkpoints_yaml = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/checkpoints.yaml"
# mock_args.config = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/c3_prod3_128px_inference_config.yaml"
mock_args.checkpoints_yaml = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/checkpoints.yaml"
mock_args.config = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/c3_prod3_128px_inference_config.yaml"
mock_args.text_prompt = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/FF_presets_structure_prompts.txt"
mock_args.structure_image = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/FF_presets_structure_images.txt"
mock_args.controls = "canny, scribble, depth"
mock_args.model_id = "prod3_27"
mock_args.limit_prompts = 30
mock_args.model_name = "prod3_27"
mock_args.save_to = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/del"
mock_args.return_file_path = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/del/file.html"
mock_args.prompt_policy = "photo_v3"
# mock_args.use_prior = 'with_text'

inference_pipeline = InferencePipeline(mock_args)

In [ ]:
# run full inference with html gen
inference_pipeline.run()

In [ ]:
# run just inference app without any MC
import torch
from clio.typings.typings import AestheticInputs
inference_pipeline.app.config.seeds = [0]
x, y = 128, 128
edge = torch.zeros((1, 2, x*8, y*8))
entity = torch.zeros((1, 3, x*8, y*8))
depth = torch.zeros((1, 1, x*8, y*8))
img = inference_pipeline.app(
    text_prompt="happy boy",
    style_prompt="",
    style_image_reference=None,
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[],
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

In [ ]:
# Run inference with MC with Scribble fix (canny working also)
import numpy as np
from PIL import Image
import torch
from clio.typings.typings import AestheticInputs

from clio.typings.interface import (
    ForwardPolicy,
    MultiConditionControls,
    PriorProcessPrompt,
    PromptProcessingPolicy,
)

# hs these are things to change
batch_size = 1
#pick just one control from ["canny", "depth", "scribble"]
inference_pipeline.controls = ["scribble"]
input_img = Image.open("/sensei-fs/users/kelkar/clio/c3p/evalsets/SM/d0.jpeg")


desired_w = inference_pipeline.app.config.dimensions.width[0] * 8
desired_h = inference_pipeline.app.config.dimensions.height[0] * 8

control_modules = []
edge = torch.zeros(
    batch_size,
    2,
    desired_h,
    desired_w,
)
depth = torch.zeros(
    batch_size,
    1,
    desired_h,
    desired_w,
)
entity = torch.zeros(
    batch_size,
    3,
    desired_h,
    desired_w,
)

w, h = input_img.size
current_ar = w / h
desired_ar = desired_w / desired_h
if current_ar > desired_ar:
    # input image is wider, crop the sides
    new_width = int(h * desired_ar)
    left = (w - new_width) // 2
    top = 0
    right = left + new_width
    bottom = h
else:
    # input image is taller, crop the top and bottom
    new_height = int(w / desired_ar)
    left = 0
    top = (h - new_height) // 2
    right = w
    bottom = top + new_height
input_img = input_img.crop((left, top, right, bottom))

if "scribble" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.EDGE)
    scribble_img = input_img.resize(
        (desired_w, desired_h), resample=Image.NEAREST
    )
    edge = torch.cat(
        [
            torch.from_numpy(np.array(scribble_img.convert("L")))
            .unsqueeze(0)
            .unsqueeze(0)
            / 255.0
        ]
        + [torch.zeros(batch_size, 1, desired_h, desired_w)],
        dim=1,
    )
    # to get scribble working you must to this:
    edge_0 = edge[:, 0, :, :]
    edge[:, 1,:,:]= edge_0

input_img = input_img.resize((desired_w, desired_h))

if "canny" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.EDGE)

    canny = inference_pipeline.canny_model(input_img, resize=False)
    canny_img = Image.fromarray(np.uint8(canny.numpy() * 255)[0])
    canny = canny.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    edge_can = torch.cat([torch.zeros_like(canny)] + [canny], dim=1)



if "depth" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.DEPTH)

    depth = inference_pipeline.depth_model(input_img, resize=False)
    depth_img = Image.fromarray(np.uint8(depth.numpy() * 255)[0])
    depth = depth.unsqueeze(0).repeat(batch_size, 1, 1, 1)



#just display the image
# canny_img.resize((128, 128))
scribble_img.resize((128, 128))
# depth_img.resize((128, 128))
inference_pipeline.app.config.seeds = [0]

img = inference_pipeline.app(
    text_prompt="happy boy",
    style_prompt=None,
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=control_modules,
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

In [ ]:
# using "prior_only" prior

#initialize with:
    # mock_args.use_prior = 'prior_only'
# and make sure that style_prompt is set to empty string
    # style_prompt=""

# in this setting we dont use the text (since priory only)

inference_pipeline.app.config.prompt_processing_config[0].prompt_policy = (
    PromptProcessingPolicy.TRAINER_V1
)
inference_pipeline.app.config.forward_policy = ForwardPolicy.IGNORE_T5

inference_pipeline.app.config.seeds = [0]
x, y = 128, 128
edge = torch.zeros((1, 2, x*8, y*8))
entity = torch.zeros((1, 3, x*8, y*8))
depth = torch.zeros((1, 1, x*8, y*8))
img = inference_pipeline.app(
    text_prompt="happy boy",
    style_prompt="",
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=control_modules,
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

In [ ]:
# using "with_text" prior

#initialize with:
    # mock_args.use_prior = 'with_text'
# and make sure that style_prompt is set to empty string
    # style_prompt=""

inference_pipeline.app.config.seeds = [0]

img = inference_pipeline.app(
    text_prompt="happy boy",
    style_prompt="",
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=control_modules,
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

In [ ]:
#Image veriations. Providing an input image

inference_pipeline.app.config.seeds = [0]
inference_pipeline.app.config.prompt_processing_config[0].prompt_policy = (
    PromptProcessingPolicy.TRAINER_V1
)
inference_pipeline.app.config.forward_policy = ForwardPolicy.IGNORE_T5

img = inference_pipeline.app(
    text_prompt=None,
    style_prompt=None,
    style_image_reference=input_img,
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=control_modules,
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

In [ ]:
## This runs dmd with MC conditions (scribble). Bakcup, dont delete
import sys
sys.path.append("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/scripts/clio3/")
from run_inference_pipelines_clio3 import InferencePipeline

# add args needed for dmd inference and load

mock_args = MockArgs()
# you can set the device if you want first gpu is 0:
mock_args.device = 1

# mock_args.checkpoints_yaml = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/checkpoints.yaml"
# mock_args.config = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/c3_prod3_128px_inference_config.yaml"
mock_args.checkpoints_yaml = "/sensei-fs/users/hshariati/experimental_configs/checkpoints_worksForDMD.yaml"
mock_args.config = "/sensei-fs/users/hshariati/experimental_configs/c3_prod3_128px_inference_config_worksForDMD.yaml"
mock_args.text_prompt = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/FF_presets_structure_prompts.txt"
mock_args.structure_image = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/FF_presets_structure_images.txt"
mock_args.controls = "canny, scribble, depth"
mock_args.model_id = "prod3_27"
mock_args.limit_prompts = 30
mock_args.model_name = "prod3_27"
mock_args.save_to = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/del"
mock_args.return_file_path = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/del/file.html"
mock_args.prompt_policy = "photo_v3"
# mock_args.use_prior = 'with_text'

inference_pipeline = InferencePipeline(mock_args)



import numpy as np
from PIL import Image
import torch
from clio.typings.typings import AestheticInputs

from clio.typings.interface import (
    ForwardPolicy,
    MultiConditionControls,
    PriorProcessPrompt,
    PromptProcessingPolicy,
)

# hs these are things to change
batch_size = 1
#pick just one control from ["canny", "depth", "scribble"]
inference_pipeline.controls = ["scribble"]
input_img = Image.open("/sensei-fs/users/kelkar/clio/c3p/evalsets/SM/d0.jpeg")


desired_w = inference_pipeline.app.config.dimensions.width[0] * 8
desired_h = inference_pipeline.app.config.dimensions.height[0] * 8

control_modules = []
edge = torch.zeros(
    batch_size,
    2,
    desired_h,
    desired_w,
)
depth = torch.zeros(
    batch_size,
    1,
    desired_h,
    desired_w,
)
entity = torch.zeros(
    batch_size,
    3,
    desired_h,
    desired_w,
)

w, h = input_img.size
current_ar = w / h
desired_ar = desired_w / desired_h
if current_ar > desired_ar:
    # input image is wider, crop the sides
    new_width = int(h * desired_ar)
    left = (w - new_width) // 2
    top = 0
    right = left + new_width
    bottom = h
else:
    # input image is taller, crop the top and bottom
    new_height = int(w / desired_ar)
    left = 0
    top = (h - new_height) // 2
    right = w
    bottom = top + new_height
input_img = input_img.crop((left, top, right, bottom))

if "scribble" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.EDGE)
    scribble_img = input_img.resize(
        (desired_w, desired_h), resample=Image.NEAREST
    )
    edge = torch.cat(
        [
            torch.from_numpy(np.array(scribble_img.convert("L")))
            .unsqueeze(0)
            .unsqueeze(0)
            / 255.0
        ]
        + [torch.zeros(batch_size, 1, desired_h, desired_w)],
        dim=1,
    )
    # to get scribble working you must to this:
    edge_0 = edge[:, 0, :, :]
    edge[:, 1,:,:]= edge_0

input_img = input_img.resize((desired_w, desired_h))

if "canny" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.EDGE)

    canny = inference_pipeline.canny_model(input_img, resize=False)
    canny_img = Image.fromarray(np.uint8(canny.numpy() * 255)[0])
    canny = canny.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    edge_can = torch.cat([torch.zeros_like(canny)] + [canny], dim=1)



if "depth" in inference_pipeline.controls:
    control_modules.append(MultiConditionControls.DEPTH)

    depth = inference_pipeline.depth_model(input_img, resize=False)
    depth_img = Image.fromarray(np.uint8(depth.numpy() * 255)[0])
    depth = depth.unsqueeze(0).repeat(batch_size, 1, 1, 1)



#just display the image
# canny_img.resize((128, 128))
scribble_img.resize((128, 128))
# depth_img.resize((128, 128))
inference_pipeline.app.config.seeds = [0]

img = inference_pipeline.app(
    text_prompt="happy boy",
    style_prompt=None,
    prior_aes=None,
    face_size=None,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0, positive_is_genai=False, avoid_aesthetic_score=2.0, avoid_is_genai=False),
    edge=edge,
    depth=depth,
    entity=entity,
    control_modules=control_modules,
    do_rag_styles=False,
    mc_injection_mask=None,
    ctype_trained_prior=False
).images
img.resize((128, 128))

## Debugging run_inference_clio3_mc.py 

In [ ]:
#script
python3 scripts/clio3/run_inference_clio3_mc.py --config "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/hshariati/clio3-mc-mar8-eval/apps/inference/inference/configs/model_explore/clio3/c3_prod3_18_128px_inference_config.yaml" --prompt "/sensei-fs/users/fchen/controlnet/test_clio-mm_v2.txt" --application "text2image_mc" --limit_prompts 10 --control "canny" --aesthetic_class 6 --return_file_path "/sensei-fs/users/hshariati/temp/out.txt" --expand_ckpt False

In [ ]:
#copy this funciton into /sensei-fs/users/hshariati/worktree_AdobeDiffusion/hshariati/clio3-mc-mar8-eval/scripts/clio3/run_inference_clio3_mc.py
def run_main(app):
    class Opt:
        pass

    opt = Opt()
    opt.prompt = "/sensei-fs/users/fchen/controlnet/test_clio-mm_v2.txt"
    opt.return_file_path = "/sensei-fs/users/hshariati/eval/clio3_dmd_guide8_mp_v4_2024_02_28-2024-03-07T11-20-43/tmp_output/out.txt"
    opt.save_to = "/sensei-fs/users/hshariati/eval/clio3_dmd_guide8_mp_v4_2024_02_28-2024-03-07T11-20-43/tmp_output"
    opt.limit_prompts = 1
    opt.control = "canny"
    opt.aesthetic_class = 6
    opt.use_adherence = True
    opt.use_ar = True
    opt.device = 0

    run_text2image_mc(
        app,
        opt.prompt,
        opt.save_to,
        opt.return_file_path,
        opt.limit_prompts,
        opt.control,
        opt.aesthetic_class,
        opt.use_adherence,
        opt.use_ar,
    )

In [ ]:
# import the functions from script
import sys
sys.path.append("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/alse/mc-extras/scripts/clio3/")
from run_inference_clio3_mc import run_main, init_app

In [ ]:
#app = init_app("/sensei-fs/users/hshariati/eval/clio3_dmd_guide8_mp_v4_2024_02_28-2024-03-07T11-20-43/tmp_configs_for_eval/G_slimmed_version_0=4000_20240312002618.yaml", 0)
# app = init_app("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/hshariati/clio3-mc-mar8-eval/apps/inference/inference/configs/model_explore/clio3/c3_prod3_18_128px_inference_config.yaml", 0)

app = init_app("/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml", 0)

In [ ]:
run_main(app)

In [ ]:
#reload config for when you get issues with aspect ratio or adherence
from pathlib import Path
from inference.inference_utils import yaml_utils
config = yaml_utils.yaml_to_inference_config(Path("/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml"))
app.config = config

In [ ]:
app.config.guidance_parameters.move_avoid_conds_for_distilled = True

## DMD related code
... and Running DMD from prod3_27 teacher (from Suresh)

In [ ]:
# Change the file permissions of the folder with your checkpoints like this:

sudo runuser -u alse -- chmod 777 -R /sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v11-alse-25/

#or
sudo runuser -u kduarte -- chmod 777 -R version_7/checkpoints/

In [ ]:
# this expands the dmd checkpotin into teacher format
import torch
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_5/checkpoints/global_step_98000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_6/checkpoints/global_step_116000_G.ckpt"
input_path = "/sensei-fs/users/siddhantg/jenkins/dmd_autoeval/checkpoints/global_step_360000_G.ckpt"
unet = torch.load(input_path, map_location="cpu")
keys = list(unet.keys())
for key in keys:
    if key.startswith("G.eps_model.D"):
        unet[key.replace("G.eps_model.D", "eps_model")] = unet.pop(key)
unet.pop("g_optimizer")
output_path = input_path.replace(".ckpt", "_slimmed.ckpt")
torch.save(unet, output_path)
print(output_path)

In [ ]:
# This is putting the T5 adaptor into the checkpoint since colligo loads it from there
import torch
import sys


# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-alse-03/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-24T10-32-39/lightning_logs/version_0/checkpoints/global_step_64000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_5/checkpoints/global_step_98000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_6/checkpoints/global_step_116000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_7/checkpoints/global_step_122000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_19/checkpoints/global_step_180000_G.ckpt"
# input_path = "/sensei-fs/projects/firefly/clio/runai_training/dmd_dmd-v12-prod3-27-84k-01/logs/clio3_dmd_guide8_mp_v4_2024_03_22-2024-03-28T14-34-13/lightning_logs/version_25/checkpoints/global_step_3040000_G.ckpt"
#same as above but copied for permission issues
input_path = "/sensei-fs/users/hshariati/del/global_step_320000_G.ckpt"
unet = torch.load(input_path, map_location="cpu")
keys = list(unet.keys())
for key in keys:
    if key.startswith("G.eps_model.D"):
        unet[key.replace("G.eps_model.D", "eps_model")] = unet.pop(key)
unet.pop("g_optimizer")
output_path = input_path.replace(".ckpt", "_slimmed.ckpt")
torch.save(unet, output_path)
print(output_path)


teacher_path = "/sensei-fs/projects/firefly/clio/training/logs/haravi/c3prod3-27-128px-2680k-aesnorm-mar3/c3prod3-27-128px-2680k-aesnorm-mar3-2024-01-12T21-41-30/lightning_logs/version_0/checkpoints/epoch=0-step=65000.ckpt"
teacher_unet = torch.load(teacher_path, map_location="cpu")

for key in teacher_unet["state_dict"].keys():
    if key in unet:
        teacher_unet["state_dict"][key] = unet[key]

output_path = input_path.replace(".ckpt", "combined_slimmed.ckpt")
torch.save(teacher_unet, output_path)
print(output_path)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from inference.inference_app import Inference
from inference.inference_md_app import InferenceMD
from inference.inference_utils import yaml_utils
from pathlib import Path
from PIL import Image
import numpy as np
import torch

In [ ]:
#load one model
# conf = "/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config.yaml" #this doesnt work
conf = "/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml" #this works
config_first = yaml_utils.yaml_to_inference_config(Path(conf))
for model_loader in config_first.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 0
app = Inference(config_first)

In [ ]:
app.config.seeds = [0]

In [ ]:
#no mc
from clio.typings.typings import AestheticInputs
x = 128
y = 128
edge = torch.zeros((1, 2, x*8, y*8))
entity = torch.zeros((1, 3, x*8, y*8))
depth = torch.zeros((1, 1, x*8, y*8))

result = app(text_prompt="some pig", edge=edge,
            aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0),
            #  content_type="vector",
depth=depth,
entity=entity,
control_modules=[]).images
result.resize((256,256))

In [ ]:
#orig from suresh with mc

import numpy as np

import torch
import random
from clio.prototyping_tools.data.annotator.entity import Entity
import torch.nn.functional as F
app.config.seeds = [333]

import pickle
import sys
sys.path.append("/sensei-fs/users/alse/dmd/dmd/implementations")

from data import ODEAndRandomBatch
d = pickle.load(open("/sensei-fs/users/alse/dmd/sample/batch_1.pkl", "rb"))

def get_edge_condition(scribble, canny, r=None):
        canny = canny.unsqueeze(1)
        w_, h_ = canny.shape[-2], canny.shape[-1]
        # reshape scribble to the same size as canny
        scribble = F.interpolate(scribble[:, None, :, :], size=(w_, h_), mode="nearest")

        edge = torch.zeros_like(torch.cat([scribble, canny], dim=1))  # [b, 2, w, h]
        for idx in range(edge.shape[0]):
            if r is None:
                r_number = random.random()
            else:
                r_number = r[idx]
            if r_number < 0.5:
                edge[idx, :1, :, :] = scribble[idx, :, :, :]
            else:
                edge[idx, 1:, :, :] = canny[idx, :, :, :]
        return edge

batch_size = len(d.image)

entity_model = Entity(target_size=1024)
entity = torch.stack([
    torch.tensor(
        np.transpose(
            entity_model.inference_mds(
                d.entity[i].to(torch.uint8).cpu().numpy(), label=d.class_labels[i]
            ),
            (2, 0, 1)
        ),
        device=0, dtype=torch.bfloat16
    ) for i in range(batch_size)])

# # app.config.iteration_parameters.num_iters = 40
# app.config.guidance_parameters.skip_cond = None
app.config.dimensions.height = [176]
app.config.dimensions.width = [88]

x, y = d.image.shape[-2:]

# edge = torch.zeros((1, 2, x*8, y*8))
# entity = torch.zeros((1, 3, x*8, y*8))
# depth = torch.zeros((1, 1, x*8, y*8))

entity = entity[0].unsqueeze(0)
depth = d.depth[0].unsqueeze(0)
edge = get_edge_condition(d.scribble, d.canny, r=[1]*batch_size)[0].unsqueeze(0)
result = app(text_prompt="a green plant with pink flowers, ctype: photo", edge=edge,
            aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0),
            #  content_type="vector",
depth=depth,
entity=entity,
control_modules=[]).images
result.resize((256,256))

In [ ]:
#imports for html saving
save_to = "/sensei-fs/users/hshariati/eval/clio3_dmd_guide8_mp_v4_2024_02_28-2024-03-07T11-20-43/tmp_output4"
return_file_path = save_to + "/.html"
# prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/eval_sentences_draw_bench.txt"
prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/new_eval_sentences_people.txt"




from clio.prototyping_tools.html.viz import HTML
from clio.prototyping_tools.aws_util import S3Handle
from PIL import Image, ImageFile
from pathlib import Path
import os
import numpy as np
import torch

def save_images(imgs, save_to, rownum, colnum):
    if not isinstance(imgs, list):
        imgs = [imgs]
    return_imgs = []
    for img in imgs:
        curimg = Image.fromarray(np.array(img))
        return_imgs.append(curimg)
        colnum += 1
    return colnum, return_imgs

def upload_dir_to_s3(save_to, return_file_path, modelname=None):
    BUCKET_NAME = "clio.corp.adobe.com"
    upload_dir_path = Path(save_to)

    s3_handle = S3Handle(
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", None),
        aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", None),
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN", None),
    )
    if not s3_handle.has_access_to_bucket(bucket_name=BUCKET_NAME):
        print("No access to s3 to upload")
        return

    s3_handle.upload_dir(
        dir_path=upload_dir_path,
        bucket=BUCKET_NAME,
        s3_key_prefix="model_evaluation",
    )
    html_title = save_to.split("/")[-1] if modelname is None else modelname
    html_url = (
        f"http://clio.corp.adobe.com/model_evaluation/{save_to}/{html_title}.html"
    )
    print(f"Results available at: {html_url}")
    if return_file_path is not None:
        with open(return_file_path, "w") as file:
            file.write(html_url)



html = HTML(web_dir=save_to, title=save_to.split("/")[-1])
html.add_table()
with open(prompt_file) as f:
    sentences = f.readlines()
sentences = [sentence.strip() for sentence in sentences]


if not os.path.isdir(save_to):
    os.makedirs(save_to)
cnt = 0

In [ ]:
# run the Display experiment and save to html
from clio.typings.typings import AestheticInputs
edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))

for sentence in sentences:
    _images = []
    _captions = []
    colnum = 0


    result_first = app(text_prompt=sentence, edge=edge,
        depth=depth,
        entity=entity,
        aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0),
        control_modules=[]).images

    colnum, return_imgs = save_images(result_first, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("dmd_1step_1000to0")

    html.add_images(
        _images,
        row_label=f"row: [{cnt}], Prompt:{sentence}",
        img_captions=_captions,
        width=min(_images[0].width, _images[0].height),
    )
    html.save()
    upload_dir_to_s3(save_to, return_file_path)
    cnt += 1

In [ ]:
# how to set prompt processing policy
from clio.typings.interface import (
    PromptProcessingPolicy,
)
app_2.config.prompt_processing_config[0].prompt_policy = PromptProcessingPolicy.PHOTO_V3

## Loading MDS dmd training data from pickle file
This is take from https://github.com/Adobe-Firefly/piat_slim/blob/main/examples/dmd_mds2html.ipynb

which creates this html: https://s3.us-west-2.amazonaws.com/kalarot-clio-data/files/test_md5_970cd166a6b48add4baf3593b63c79c5.html

In [ ]:
from PIL import Image
import numpy as np
import torch
import copy
def debatch(data):
    if 'exif' in data:
        for index, tensor in enumerate(data.pop('exif')):
            data[f'exif_{index}'] = tensor.tolist()
    return [{key: value[i] for key, value in data.items()} for i in range(len(next(iter(data.values()))))]

def debatch(data):
    data_copy = copy.deepcopy(data)
    for idx in range(len(data_copy)):
        for key in data_copy[idx].keys():
            if isinstance(data_copy[idx][key], torch.Tensor) and data_copy[idx][key].dim() > 0 and data_copy[idx][key].size(0) == 1:
                data_copy[idx][key] = data_copy[idx][key][0]
    return data_copy


def entity2pil(entity):
    rng = np.random.default_rng(42)
    cmap = rng.integers(0, 256, size=(256, 3), dtype=np.uint8)
    cmap[0] = 0
    return Image.fromarray(cmap[entity])

def tensor2pil(tensor):
    if tensor.dim() == 4 or tensor.shape[0] == 1:
        tensor = tensor[0]
    if tensor.shape[0] == 12:
        tensor = torch.clamp((tensor[0:3, :, :] + 1.0) / 2.0, 0.0, 1.0)
    if tensor.dim() == 2:
        tensor = tensor.unsqueeze(0).repeat(3, 1, 1)
    if torch.is_floating_point(tensor):
        tensor = tensor * 255
    tensor = tensor.byte()
    assert(tensor.dim() == 3 and tensor.shape[0] == 3)
    return Image.fromarray(tensor.permute(1, 2, 0).numpy(), 'RGB')

In [ ]:
import pickle
#https://s3.us-west-2.amazonaws.com/kalarot-clio-data/files/test_md5_f920f709e0a5e313703d20a68577a264.html
#with open('/sensei-fs/projects/firefly/clio/vae_inference/sample.pkl', 'rb') as f:
with open('/sensei-fs/users/alse/dmd/sample/dmd_phase1_5/samples_100.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
datas= debatch(data)
datas[0].keys()

In [ ]:
#you can convert like this
tensor2pil(data[0]['latent'][0]).resize((256,256))
tensor2pil(data[0]['image']).resize((256,256))
tensor2pil(datas[0]['canny']).resize((256,256))
entity2pil(datas[0]['entity']).resize((256,256))

In [ ]:
img = tensor2pil(data[0]['image'])
img

In [ ]:
datas[0]['aesthetic_score'].item()


In [ ]:
def convert_depth(can):
    k = can.unsqueeze(0)
    value = torch.stack(
        [torch.zeros_like(k[0])] + [k[0]],  # type: ignore
    )
    value = value.unsqueeze(0)
    return value

In [ ]:
datas[1]['canny'].shape

In [ ]:
k = datas[1]['canny'].sum()
k

In [ ]:
value = torch.stack(
    [torch.zeros_like(k[0])] + [k[0]],  # type: ignore
)
value = value.unsqueeze(0)
value.shape

In [ ]:
value.sum()

In [ ]:
value.shape

In [ ]:
datas[1]['scribble'].sum()

In [ ]:
datas[1]['ode_scribble_vs_canny_prob']

In [ ]:
                elif key == "mdsCannyEdge":
                    # value = get_edge_condition(mds["mdsScribble"], mds["mdsCannyEdge"])
                    value = torch.stack(
                        [torch.zeros_like(mds["mdsCannyEdge"][0])] + [mds["mdsCannyEdge"][0]],  # type: ignore
                        # [torch.zeros_like(mds["mdsCannyEdge"])] + [mds["mdsCannyEdge"]],  # type: ignore
                    )
                    value = value.unsqueeze(0)
                elif key == "mdsScribble":
                    value = torch.stack(
                        [torch.zeros_like(mds["mdsScribble"][0])] + [mds["mdsScribble"][0]],  # type: ignore
                    )
                    value = value.unsqueeze(0)
                elif key == "mdsDepthMap":
                    value = value.unsqueeze(0)
                elif key == "mdsEntity":
                    entity = Entity(
                        target_size=value.shape
                    )  # TODO: if target_size actually gets ignored then this can be done once in init
                    dtype = value.dtype
                    value = np.transpose(
                        entity.inference_mds(
                            value.cpu().numpy(), label=mds["strClassLabels"][0]
                        ),
                        (2, 0, 1),
                    )
                    value = torch.from_numpy(value).unsqueeze(0).to(dtype)
                setattr(self, key, value)

# official experiments

## Experiment 03/20/2024:
    Run the DMD with MC from Suresh as first stage. From 1000 to 0. Then pass it to MOE 2nd stage for 500 to 0. 
    
 ### env:
/sensei-fs/users/hshariati/workspace/AdobeDiffusion

docker: 	docker-clio-release.dr-uw2.adobeitc.com/inference/

clio-inference-cuda12:0.1.36_2.9.0

machine: dmd-mc-test, clio_model_efficiency, cluster 4


In [ ]:
#imports
from inference.inference_app import Inference
from inference.inference_md_app import InferenceMD
from inference.inference_utils import yaml_utils
from pathlib import Path
from PIL import Image
import numpy as np
import torch

In [ ]:
#load the dmd model on device 0
config = yaml_utils.yaml_to_inference_config(
                Path("/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml")
            )


for model_loader in config.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 0

app = Inference(config)


In [ ]:
#load the second model only on gpu2
config_second = yaml_utils.yaml_to_inference_config(Path("/sensei-fs/users/hshariati/experimental_configs/c3_prod3_27_prod3_16_moe_128px_inference_config_second.yaml"))
for model_loader in config_second.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 1
app_second = Inference(config_second)

In [ ]:
#set both seeds
app.config.seeds = [333]
app_second.config.seeds = [333]

In [ ]:
print(app_second.config.prompt_processing_config[0].prompt_policy, app_second.config.iteration_parameters.num_iters, app_second.config.sampling_parameters[0].timesteps_start_ratio,  app_second.config.sampling_parameters[0].timesteps_end_ratio)

In [ ]:
# run the dmd model.
from clio.typings.typings import AestheticInputs
app.config.seeds = [0]

edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))

result = app(text_prompt="a dog with a hat, sharp, photo-realistic", edge=edge,
    depth=depth,
    entity=entity,
    edege=edge,
    aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0),
    control_modules=[]).images
result

In [ ]:
# run the second model
result_second = app_second(text_prompt="a dog with a hat, sharp, photo-realistic | ctype: photo",initial_image=result, edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[]).images

In [ ]:
#imports for html saving
save_to = "/sensei-fs/users/hshariati/dmd_moe_html7"
return_file_path = save_to + "/.html"
# prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/eval_sentences_draw_bench.txt"
prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/new_eval_sentences_people.txt"




from clio.prototyping_tools.html.viz import HTML
from clio.prototyping_tools.aws_util import S3Handle
from PIL import Image, ImageFile
from pathlib import Path
import os
import numpy as np
import torch

def save_images(imgs, save_to, rownum, colnum):
    if not isinstance(imgs, list):
        imgs = [imgs]
    return_imgs = []
    for img in imgs:
        curimg = Image.fromarray(np.array(img))
        return_imgs.append(curimg)
        colnum += 1
    return colnum, return_imgs

def upload_dir_to_s3(save_to, return_file_path, modelname=None):
    BUCKET_NAME = "clio.corp.adobe.com"
    upload_dir_path = Path(save_to)

    s3_handle = S3Handle(
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", None),
        aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", None),
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN", None),
    )
    if not s3_handle.has_access_to_bucket(bucket_name=BUCKET_NAME):
        print("No access to s3 to upload")
        return

    s3_handle.upload_dir(
        dir_path=upload_dir_path,
        bucket=BUCKET_NAME,
        s3_key_prefix="model_evaluation",
    )
    html_title = save_to.split("/")[-1] if modelname is None else modelname
    html_url = (
        f"http://clio.corp.adobe.com/model_evaluation/{save_to}/{html_title}.html"
    )
    print(f"Results available at: {html_url}")
    if return_file_path is not None:
        with open(return_file_path, "w") as file:
            file.write(html_url)



html = HTML(web_dir=save_to, title=save_to.split("/")[-1])
html.add_table()
with open(prompt_file) as f:
    sentences = f.readlines()
sentences = [sentence.strip() for sentence in sentences]


if not os.path.isdir(save_to):
    os.makedirs(save_to)
cnt = 0

In [ ]:
# run the Display experiment and save to html
from clio.typings.typings import AestheticInputs
edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))
app_second.config.latent_inference_parameters.encode_initial_image = True

for sentence in sentences:
    _images = []
    _captions = []
    colnum = 0

    # result_first = app(text_prompt=sentence, edge=edge,
    #     depth=depth,
    #     entity=entity,
    #     content_type="photo",
    #     control_modules=[]).images

    result_first = app(text_prompt=sentence, edge=edge,
        depth=depth,
        entity=entity,
        aesthetic_inputs=AestheticInputs(positive_aesthetic_score=6.0),
        control_modules=[]).images

    colnum, return_imgs = save_images(result_first, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("dmd_1step_1000to0")


    result_second = app_second(text_prompt=sentence + " | ctype: photo",initial_image=result_first, edge=edge,
        depth=depth,
        entity=entity,
        control_modules=[]).images


    colnum, return_imgs = save_images(result_second, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("stage2_15_steps_500to0")

    html.add_images(
        _images,
        row_label=f"row: [{cnt}], Prompt:{sentence}",
        img_captions=_captions,
        width=min(_images[0].width, _images[0].height),
    )
    html.save()
    upload_dir_to_s3(save_to, return_file_path)
    cnt += 1

## Experiment 03/20/2024:
    Run moe 1st stage from 1000 to 0 (full) then use this as a display. 
    On a mouse up event, run the second stage from 500 to 0. Similar to SDEdit. 
    The goal is to run this with DMD instead of with MOE so that preview is fast.
    
 ### env:
/sensei-fs/users/hshariati/workspace/AdobeDiffusion

docker: 	docker-clio-release.dr-uw2.adobeitc.com/inference/

clio-inference-cuda12:0.1.36_2.9.0

machine: dmd-mc-test, clio_model_efficiency, cluster 4


In [ ]:
#imports
from inference.inference_app import Inference
from inference.inference_md_app import InferenceMD
from inference.inference_utils import yaml_utils
from pathlib import Path
from PIL import Image
import numpy as np
import torch

In [ ]:
#just for test: running both in Moe together

config_path = Path("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/apps/inference/inference/configs/model_explore/clio3/prod/c3_prod3_27_prod3_16_moe_128px_inference_config.yaml")
# "/sensei-fs/users/alse/dmd/dmd/implementations/models/dmd_prod3_18_inference_config.yaml"
# "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/hshariati/clio3-mc-mar8-eval/apps/inference/inference/configs/model_explore/clio3/c3_prod3_18_prod3_16_moe_128px_inference_config.yaml"

config = yaml_utils.yaml_to_inference_config(config_path)

for model_loader in config.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 0

app = Inference(config)
app.config.seeds = [0]
app.config.dimensions.height = [128, 128]
app.config.dimensions.width = [128, 128]
edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))
result = app(text_prompt="dog with a hat", edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[]).images
result

In [ ]:
#load the first model only on gpu1
config_first = yaml_utils.yaml_to_inference_config(Path("/sensei-fs/users/hshariati/experimental_configs/c3_prod3_27_prod3_16_moe_128px_inference_config_first_just.yaml"))
for model_loader in config_first.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 0
app_first = Inference(config_first)

In [ ]:
#load the second model only on gpu2
config_second = yaml_utils.yaml_to_inference_config(Path("/sensei-fs/users/hshariati/experimental_configs/c3_prod3_27_prod3_16_moe_128px_inference_config_second.yaml"))
for model_loader in config_second.model_configs:
                if model_loader[1] is not None:
                    model_loader[1].device = 1
app_second = Inference(config_second)

In [ ]:
#run the first model
edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))
result_first = app_first(text_prompt="dog with a hat", edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[]).images

result_first.resize((256,256))

In [ ]:
#run the second model
app_second.config.latent_inference_parameters.encode_initial_image = True

edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))
result_second = app_second(text_prompt="dog with a hat",initial_image=result_first, edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[]).images

result_second.resize((256,256))

In [ ]:
#just for test: run media save for test
import imageio
help(imageio.mimwrite)
result_second = app_second.export_media(output_file_name="/sensei-fs/users/hshariati/worktree_AdobeDiffusion/temp/output.mp4",text_prompt="dog with a hat",initial_image=result_first, edge=edge,
    depth=depth,
    entity=entity,
    control_modules=[]).images


In [ ]:
#imports for html saving
from clio.prototyping_tools.html.viz import HTML
from clio.prototyping_tools.aws_util import S3Handle
from PIL import Image, ImageFile
from pathlib import Path
import os
import numpy as np
import torch

def save_images(imgs, save_to, rownum, colnum):
    if not isinstance(imgs, list):
        imgs = [imgs]
    return_imgs = []
    for img in imgs:
        curimg = Image.fromarray(np.array(img))
        return_imgs.append(curimg)
        colnum += 1
    return colnum, return_imgs

def upload_dir_to_s3(save_to, return_file_path, modelname=None):
    BUCKET_NAME = "clio.corp.adobe.com"
    upload_dir_path = Path(save_to)

    s3_handle = S3Handle(
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", None),
        aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", None),
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN", None),
    )
    if not s3_handle.has_access_to_bucket(bucket_name=BUCKET_NAME):
        print("No access to s3 to upload")
        return

    s3_handle.upload_dir(
        dir_path=upload_dir_path,
        bucket=BUCKET_NAME,
        s3_key_prefix="model_evaluation",
    )
    html_title = save_to.split("/")[-1] if modelname is None else modelname
    html_url = (
        f"http://clio.corp.adobe.com/model_evaluation/{save_to}/{html_title}.html"
    )
    print(f"Results available at: {html_url}")
    if return_file_path is not None:
        with open(return_file_path, "w") as file:
            file.write(html_url)



save_to = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/temp2"
return_file_path = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/temp2/.html"
html = HTML(web_dir=save_to, title=save_to.split("/")[-1])
html.add_table()
# prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/AdobeDiffusion/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/eval_sentences_draw_bench.txt"
prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/kduarte/dmd-sliders/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/eval_sentences_test.txt"
with open(prompt_file) as f:
    sentences = f.readlines()
sentences = [sentence.strip() for sentence in sentences]


if not os.path.isdir(save_to):
    os.makedirs(save_to)
cnt = 0

In [ ]:
# run the Display experiment and save to html
edge = torch.zeros((1, 2, 128*8, 128*8))
entity = torch.zeros((1, 3, 128*8, 128*8))
depth = torch.zeros((1, 1, 128*8, 128*8))
app_second.config.latent_inference_parameters.encode_initial_image = True

for sentence in sentences:
    _images = []
    _captions = []
    colnum = 0

    result_first = app_first(text_prompt=sentence, edge=edge,
        depth=depth,
        entity=entity,
        control_modules=[]).images

    colnum, return_imgs = save_images(result_first, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("stage1_40_steps_1000to0")


    result_second = app_second(text_prompt=sentence,initial_image=result_first, edge=edge,
        depth=depth,
        entity=entity,
        control_modules=[]).images


    colnum, return_imgs = save_images(result_second, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("stage2_15_steps_500to0")

    html.add_images(
        _images,
        row_label=f"row: [{cnt}], Prompt:{sentence}",
        img_captions=_captions,
        width=min(_images[0].width, _images[0].height),
    )
    html.save()
    upload_dir_to_s3(save_to, return_file_path)
    cnt += 1

In [ ]:
import sys
sys.path.append("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/alse/mc-extras/scripts/clio3/")
from run_inference_clio3_mc import run_main, init_app

In [ ]:
#app = init_app("/sensei-fs/users/hshariati/eval/clio3_dmd_guide8_mp_v4_2024_02_28-2024-03-07T11-20-43/tmp_configs_for_eval/G_slimmed_version_0=4000_20240312002618.yaml", 0)
# app = init_app("/sensei-fs/users/hshariati/worktree_AdobeDiffusion/hshariati/clio3-mc-mar8-eval/apps/inference/inference/configs/model_explore/clio3/c3_prod3_18_128px_inference_config.yaml", 0)

app = init_app("/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml", 0)

In [ ]:
run_main(app)

In [ ]:
#reload config for when you get issues with aspect ratio or adherence
from pathlib import Path
from inference.inference_utils import yaml_utils
config = yaml_utils.yaml_to_inference_config(Path("/sensei-fs/users/hshariati/experimental_configs/dmd_prod3_18_inference_config_newCkpt.yaml"))
app.config = config

In [ ]:
app.config.guidance_parameters.move_avoid_conds_for_distilled = True

## Experiment orignal SDEdit:
    Run the student (DMD), then pass the results as initial image to teacher (AIT), add some noise and run for a few steps. 
    Goal: The student runs in one step, but the results are not great. The teacher usually in 40, but the results are great. Can we run the student (1 step) and then pass that as initial image to teacher and run it for a few steps and get good results?
### env:
/sensei-fs/users/hshariati/clio-engine/AdobeDiffusion
* kduarte/dmd-sliders

commit a0da83304e7b39a3bf911f067321679c7f0066ee (HEAD -> kduarte/dmd-sliders, origin/kduarte/dmd-sliders)
Merge: 421819504 9d5ad144d
Author: kduarte <kduarte@adobe.com>
Date:   Thu Feb 15 21:50:37 2024 +0000

    fixed interface conflict

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pkg_resources
from inference.inference_utils import yaml_utils
from inference.inference_app import Inference
import torch

# dmd_config_yaml = "/sensei-fs/users/kduarte/dmd/AdobeDiffusion/apps/inference/inference/configs/model_explore/dmd/dmd_inference_config.yaml"
# dmd_config_yaml = "/sensei-fs/users/kduarte/dmd/AdobeDiffusion/apps/inference/inference/configs/model_explore/dmd/dmd_inference_config_ait.yaml"

#this has anchor cross att which gives better results (its what we use for demo)
dmd_config_yaml = "/sensei-fs/users/kduarte/dmd/AdobeDiffusion/apps/inference/inference/configs/model_explore/dmd/dmd_inference_config_att_comp.yaml"
dmd_config = yaml_utils.yaml_to_inference_config(dmd_config_yaml)
dmd_config.seeds = [0]


teacher_config_yaml = "/sensei-fs/users/alse/project-noise-bank/clio3_config_AIT.yaml"
teacher_config = yaml_utils.yaml_to_inference_config(teacher_config_yaml)
teacher_config.seeds = [0]

In [ ]:
app = Inference(dmd_config)
app.eps_model = None
torch.cuda.empty_cache()

from inference.inference_utils.model_loading_utils import initialize_eps_model
dmd_eps_model, dmd_device = initialize_eps_model(dmd_config)
teacher_eps_model, teache_device = initialize_eps_model(teacher_config)

In [ ]:
from clio.prototyping_tools.html.viz import HTML
from clio.prototyping_tools.aws_util import S3Handle
from PIL import Image, ImageFile
from pathlib import Path
import os
import numpy as np
import torch

def save_images(imgs, save_to, rownum, colnum):
    if not isinstance(imgs, list):
        imgs = [imgs]
    return_imgs = []
    for img in imgs:
        curimg = Image.fromarray(np.array(img))
        return_imgs.append(curimg)
        colnum += 1
    return colnum, return_imgs

def upload_dir_to_s3(save_to, return_file_path, modelname=None):
    BUCKET_NAME = "clio.corp.adobe.com"
    upload_dir_path = Path(save_to)

    s3_handle = S3Handle(
        aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", None),
        aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", None),
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN", None),
    )
    if not s3_handle.has_access_to_bucket(bucket_name=BUCKET_NAME):
        print("No access to s3 to upload")
        return

    s3_handle.upload_dir(
        dir_path=upload_dir_path,
        bucket=BUCKET_NAME,
        s3_key_prefix="model_evaluation",
    )
    html_title = save_to.split("/")[-1] if modelname is None else modelname
    html_url = (
        f"http://clio.corp.adobe.com/model_evaluation/{save_to}/{html_title}.html"
    )
    print(f"Results available at: {html_url}")
    if return_file_path is not None:
        with open(return_file_path, "w") as file:
            file.write(html_url)



save_to = "/sensei-fs/users/hshariati/sdedit"
return_file_path = "/sensei-fs/users/hshariati/sdedit/.html"
html = HTML(web_dir=save_to, title=save_to.split("/")[-1])
html.add_table()
prompt_file = "/sensei-fs/users/hshariati/worktree_AdobeDiffusion/kduarte/dmd-sliders/cliocore/prototyping_tools/src/clio/prototyping_tools/data/eval_sentences/tests/eval_sentences.txt"
with open(prompt_file) as f:
    sentences = f.readlines()
sentences = [sentence.strip() for sentence in sentences]

slider_direction = torch.load("/sensei-fs/users/kduarte/dmd/AdobeDiffusion/data/weight_direction.pt")

if not os.path.isdir(save_to):
    os.makedirs(save_to)
cnt = 0

run_low_noise = [0.3, 0.35, 0.4, 0.45, 0.5]
run_low_steps = [2, 5]
run_high_noise = [0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
run_hight_steps = [7, 10]

In [ ]:
for sentence in sentences:
    _images = []
    _captions = []
    colnum = 0

    app.config = dmd_config
    app.eps_model = dmd_eps_model
    app.config.iteration_parameters.num_iters = 1
    app.config.sampling_parameters[0].timesteps_start_ratio = 1.0
    app.config.latent_inference_parameters.encode_initial_image = False
    app.config.sampling_parameters[0].timesteps_end_ratio = 0.0

    result_dmd = app(
        text_prompt = sentence,
        avoid_prompt="",
        slider_direction=slider_direction,
        slider_intensity=0
    )

    colnum, return_imgs = save_images(result_dmd.images, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("student")


    app.config = teacher_config
    app.eps_model = teacher_eps_model
    app.config.iteration_parameters.num_iters = 40
    app.config.sampling_parameters[0].timesteps_start_ratio = 1.0
    app.config.latent_inference_parameters.encode_initial_image = False
    app.config.sampling_parameters[0].timesteps_end_ratio = 0.0

    res = app(
        text_prompt=sentence,
    )

    colnum, return_imgs = save_images(res.images, save_to, cnt, colnum)
    _images += return_imgs
    _captions.append("teacher")


    for noise in run_high_noise:
        for steps in run_hight_steps:
            app.config = teacher_config
            app.eps_model = teacher_eps_model
            app.config.iteration_parameters.num_iters = steps
            app.config.sampling_parameters[0].timesteps_start_ratio = noise
            app.config.latent_inference_parameters.encode_initial_image = True
            app.config.sampling_parameters[0].timesteps_end_ratio = 0.0

            res = app(
                text_prompt=sentence,
                initial_image=result_dmd.images,
            )

            colnum, return_imgs = save_images(res.images, save_to, cnt, colnum)
            _images += return_imgs
            ts = f"Noise: {noise}, steps: {steps}"
            _captions.append(ts)


    for noise in run_low_noise:
        for steps in run_low_steps:
            app.config = teacher_config
            app.eps_model = teacher_eps_model
            app.config.iteration_parameters.num_iters = steps
            app.config.sampling_parameters[0].timesteps_start_ratio = noise
            app.config.latent_inference_parameters.encode_initial_image = True
            app.config.sampling_parameters[0].timesteps_end_ratio = 0.0

            res = app(
                text_prompt=sentence,
                initial_image=result_dmd.images,
            )

            colnum, return_imgs = save_images(res.images, save_to, cnt, colnum)
            _images += return_imgs
            ts = f"Noise: {noise}, steps: {steps}"
            _captions.append(ts)


    html.add_images(
        _images,
        row_label=f"row: [{cnt}], Prompt:{sentence}",
        img_captions=_captions,
        width=min(_images[0].width, _images[0].height),
    )
    html.save()
    upload_dir_to_s3(save_to, return_file_path)
    cnt += 1

